创建人工数据集并存储在csv文件中

In [1]:
import os
import pandas as pd
os.makedirs(os.path.join('..','data'), exist_ok=True)  # data目录存在也不会报错
data_file = os.path.join('..','data','house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n') # 列名
    f.write('NA,Pave,127500\n') # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


处理缺失值

考虑插值法，通过位置索引iloc，把data分成inputs和outputs，其中前者为data的前两列，后者为data的第最后一列，对于inputs中缺少的数值，用同一列中的均值去代替。这里也可以使用loc（知道列的名字，按照名称选择）

In [ ]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]  # 这里用的是位置索引，只知道列的位置，不知道列的名字
inputs = inputs.fillna(inputs.mean())  # 这个地方对于pandas的新版本，不会报错，只会填充数值的列
print(inputs)

TypeError: can only concatenate str (not "int") to str

对于inputs中的NaN值，视NaN值为一个类别，对于Alley类型只接受两种类型的值，即Pave和NA，pandas会自动转换为两列"Alley_Pave"和"Alley_nan"

In [ ]:
inputs = pd.get_dummies(inputs, dummy_na=True)  # dummy是虚拟变量，把一种类别型数据转换为数值型数据
# dummy_na=True表示把NaN值也转换为一个类别，即"Alley_nan"
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       NaN        True      False
1       2.0       False       True
2       4.0       False       True
3       NaN       False       True


转换为张量的格式

In [4]:
import torch
x = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
x, y

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(tensor([[nan, 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [nan, 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))